## はじめに

このノートブックでは、GlacierStyle ECサイトのデータを Snowflake に取り込み、**raw 層**のテーブルを作成します。音声や PDF のような非構造化データも、ステージに置けば SQL でテーブルにできることを体験します。

**前提条件:**
- `setup_20261216.sql` が実行済みであること（データベース・スキーマ・ステージ・GitHub 連携の設定と、データファイルのコピーが完了）

**主な処理内容:**
1. 構造化データ（CSV）のロード
2. 音声ファイルの文字起こし（AI_TRANSCRIBE）
3. PDF ドキュメントの解析（AI_PARSE_DOCUMENT）

**生成されるテーブル（raw 層）:**

*構造化データ*
- `PRODUCTS`: 商品マスタ（55件）
- `CUSTOMERS`: 顧客マスタ（300件）
- `ORDERS`: 注文データ（2,913件。ロット番号付き）
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みの通話（88件）

*非構造化データから作成*
- `CALL_TRANSCRIPTS`: 文字起こし済みの通話88件と、音声10件の文字起こし結果を合わせた通話テキスト（98件）
- `DOC_PARSED`: PDF 11ファイルをページ単位で解析したテキスト（99ページ）

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。このノートブックの AI 処理は、音声10件と PDF 99ページです。

> **途中で遅れた場合**: `checkpoint_20261216.sql` の「Part 1」を実行すると、このノートブックの完了時点のテーブルを復元できます。

In [ ]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するロール、ウェアハウス、スキーマを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part1/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 1. 構造化データのロード

ステージにコピーした CSV を、`COPY INTO` でテーブルに読み込みます。CSV の1行目（ヘッダー）の列名とテーブルの列名を対応させて読み込むため、列の順番を気にする必要はありません。

`ORDERS` の `lot_no` は商品の入荷ロットです。Part 2 で、通話の内容とこの列を組み合わせて分析します。

In [ ]:
%%sql -r result_load_csv
-- ============================================================================
-- テーブル定義の作成と CSV のロード
-- ============================================================================
-- CSV のヘッダー行を列名として使うファイル形式
CREATE OR REPLACE FILE FORMAT HANDSON_CSV
    TYPE = CSV
    PARSE_HEADER = TRUE
    FIELD_OPTIONALLY_ENCLOSED_BY = '"';

-- 商品マスタ
CREATE OR REPLACE TABLE PRODUCTS (
    product_id VARCHAR, product_name VARCHAR,
    category_l1 VARCHAR, category_l2 VARCHAR, category_l3 VARCHAR,
    brand VARCHAR, list_price NUMBER, cost_price NUMBER, launch_date DATE
);

-- 顧客マスタ
CREATE OR REPLACE TABLE CUSTOMERS (
    customer_id VARCHAR, last_name VARCHAR, first_name VARCHAR, gender VARCHAR,
    birth_date DATE, email VARCHAR, phone VARCHAR,
    prefecture VARCHAR, city VARCHAR, registration_date DATE, membership_tier VARCHAR
);

-- 注文データ（lot_no は商品の入荷ロット）
CREATE OR REPLACE TABLE ORDERS (
    order_id VARCHAR, order_datetime TIMESTAMP_NTZ, customer_id VARCHAR, product_id VARCHAR,
    quantity NUMBER, unit_price NUMBER, discount_amount NUMBER, total_amount NUMBER,
    order_channel VARCHAR, order_status VARCHAR, lot_no VARCHAR
);

-- 文字起こし済みの通話
CREATE OR REPLACE TABLE RAW_CALL_TRANSCRIPTS (
    call_id VARCHAR, call_datetime TIMESTAMP_NTZ, customer_id VARCHAR, order_id VARCHAR,
    product_id VARCHAR, agent_id VARCHAR, call_duration_sec NUMBER, transcript_text VARCHAR
);

-- CSV の列名とテーブルの列名を対応させてロード
COPY INTO PRODUCTS FROM @DATA_STAGE FILES = ('products.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO CUSTOMERS FROM @DATA_STAGE FILES = ('customers.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO ORDERS FROM @DATA_STAGE FILES = ('orders.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO RAW_CALL_TRANSCRIPTS FROM @DATA_STAGE FILES = ('call_transcripts.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
%%sql -r result_check_csv
-- ============================================================================
-- ロード結果の確認（文字起こし済みの通話を3件表示）
-- ============================================================================
SELECT call_id, call_datetime, product_id, LEFT(transcript_text, 80) AS transcript_preview
FROM RAW_CALL_TRANSCRIPTS
ORDER BY call_id
LIMIT 3;

## 2. 音声ファイルの文字起こし

コールセンターの通話音声（MP3、10件）を `AI_TRANSCRIBE` で文字起こしします。ファイルは `TO_FILE('@ステージ', 'パス')` で指定します。

この10件は文字起こしを体験するためのサンプルで、注文や商品のデータとはつながっていません。Part 2 以降の分析には、すでに文字起こし済みの88件とあわせて使います。

In [ ]:
display_image('ai_transcribe.png')

In [ ]:
%%sql -r result_try_transcribe
-- ============================================================================
-- AI_TRANSCRIBE: 1件だけ文字起こしを試してみよう
-- ============================================================================
-- 結果は JSON で返り、text に文字起こし、audio_duration に音声の長さ（秒）が入ります
SELECT AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', 'voice_logs/CALL_001.mp3')) AS transcribe_result;

In [ ]:
%%sql -r result_call_transcripts
-- ============================================================================
-- 音声10件を文字起こしし、文字起こし済みの88件と合わせる
-- ============================================================================
-- DIRECTORY(@DATA_STAGE) でステージ上のファイル一覧を取得し、MP3 だけを処理します
CREATE OR REPLACE TABLE CALL_TRANSCRIPTS AS
SELECT
    call_id, call_datetime, customer_id, order_id, product_id,
    transcript_text,
    '文字起こし済み' AS source
FROM RAW_CALL_TRANSCRIPTS
UNION ALL
SELECT
    'VOICE-' || REPLACE(SPLIT_PART(relative_path, '/', -1), '.mp3', '') AS call_id,
    NULL::TIMESTAMP_NTZ, NULL, NULL, NULL,  -- 音声のサンプルは注文・商品とつながっていない
    AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', relative_path)):text::VARCHAR AS transcript_text,
    '音声' AS source
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path ILIKE 'voice_logs/%.mp3';

-- 件数を出どころごとに確認（文字起こし済み 88件、音声 10件）
SELECT source, COUNT(*) AS record_count
FROM CALL_TRANSCRIPTS
GROUP BY source;

## 3. PDF ドキュメントの解析

FAQ、運用マニュアル、品質報告書、通達、店舗報告などの PDF（11ファイル）を `AI_PARSE_DOCUMENT` で解析します。

- `'mode': 'LAYOUT'`: 見出しや表の構造を保ったまま、Markdown 形式のテキストにします
- `'page_split': true`: ページごとに結果を分けます。後で「どのページに書いてあるか」を残せます

In [ ]:
display_image('ai_parse_document.png')

In [ ]:
%%sql -r result_try_parse
-- ============================================================================
-- AI_PARSE_DOCUMENT: 品質報告書（1ページ）の解析を試してみよう
-- ============================================================================
-- pages の各要素の content に、ページの本文が Markdown で入ります
SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@DATA_STAGE', 'documents/quality_report_202611.pdf'),
    {'mode': 'LAYOUT', 'page_split': true}
) AS parse_result;

In [ ]:
%%sql -r result_doc_parsed
-- ============================================================================
-- PDF 11ファイルをページ単位でテーブルにする
-- ============================================================================
CREATE OR REPLACE TABLE DOC_PARSED AS
WITH parsed AS (
    SELECT
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'mode': 'LAYOUT', 'page_split': true}
        ) AS doc
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE '%.pdf'
)
-- pages の配列を FLATTEN で1ページ1行に展開
SELECT
    p.file_name,
    pg.index + 1 AS page_no,
    pg.value:content::VARCHAR AS page_text
FROM parsed p,
LATERAL FLATTEN(input => p.doc:pages) pg;

-- ファイルごとのページ数と、1ページ目の冒頭を確認
SELECT
    file_name,
    COUNT(*) AS page_count,
    LEFT(MAX(IFF(page_no = 1, page_text, NULL)), 60) AS first_page_preview
FROM DOC_PARSED
GROUP BY file_name
ORDER BY file_name;

## 4. データ確認

作成した raw 層のテーブルの件数を確認します。

In [ ]:
%%sql -r result_verify
-- ============================================================================
-- 各テーブルのレコード数を確認
-- ============================================================================
SELECT 'PRODUCTS' AS table_name, COUNT(*) AS record_count, '商品マスタ（55件）' AS description FROM PRODUCTS
UNION ALL
SELECT 'CUSTOMERS', COUNT(*), '顧客マスタ（300件）' FROM CUSTOMERS
UNION ALL
SELECT 'ORDERS', COUNT(*), '注文データ（2,913件）' FROM ORDERS
UNION ALL
SELECT 'RAW_CALL_TRANSCRIPTS', COUNT(*), '文字起こし済みの通話（88件）' FROM RAW_CALL_TRANSCRIPTS
UNION ALL
SELECT 'CALL_TRANSCRIPTS', COUNT(*), '通話テキスト（98件）' FROM CALL_TRANSCRIPTS
UNION ALL
SELECT 'DOC_PARSED', COUNT(*), 'PDF のページ（99ページ）' FROM DOC_PARSED;

## まとめ

このノートブックでは、構造化データと非構造化データを取り込み、raw 層のテーブルを作成しました。

### 作成したテーブル一覧（raw 層）

**構造化データ**
- `PRODUCTS`: 商品マスタ
- `CUSTOMERS`: 顧客マスタ
- `ORDERS`: 注文データ
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みの通話

**非構造化データから作成**
- `CALL_TRANSCRIPTS`: 通話テキスト（音声の文字起こしを含む）
- `DOC_PARSED`: PDF のページ単位のテキスト

### 使用した Cortex AI 関数

- `AI_TRANSCRIBE`: 音声ファイルの文字起こし
- `AI_PARSE_DOCUMENT`: PDF ドキュメントの解析

### 次のステップ

- **Part 2**: raw 層のテキストから、AI 関数で問い合わせ種別・感情・解約リスクなどを取り出し、分析や検索に使える conform 層のテーブルを作成します